# Revarie LM v1.0 – Nightly Persona Consolidation
This notebook runs daily at 1:00 AM IST to consolidate the previous day's interactions
and fine-tune the LoRA adapters for Samara and Artery.

**Theoretical Foundation:**
- Stickgold & Walker (2013): Sleep-dependent memory consolidation
- McClelland et al. (1995): Complementary learning systems
- Hu et al. (2021): LoRA fine-tuning

In [ ]:
!pip install -q torch transformers peft datasets aiohttp safetensors

import os
import json
import torch
import asyncio
import aiohttp
import numpy as np
from datetime import datetime, timedelta
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
import safetensors.torch

In [ ]:
# =============================================================================
# CONFIGURATION (Load from environment)
# =============================================================================
VAULT_API_URL = os.environ.get("VAULT_API_URL")
VAULT_API_KEY = os.environ.get("VAULT_API_KEY")
HF_TOKEN = os.environ.get("HF_TOKEN")
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

ist_now = datetime.utcnow() + timedelta(hours=5, minutes=30)
yesterday = (ist_now - timedelta(days=1)).strftime("%Y-%m-%d")
print(f"🔄 Running consolidation for {yesterday}")

In [ ]:
async def fetch_sessions(date: str):
    """Fetch all sessions for a given date from D1."""
    headers = {"x-api-key": VAULT_API_KEY}
    url = f"{VAULT_API_URL}/sessions/{date}"
    async with aiohttp.ClientSession() as session:
        async with session.get(url, headers=headers) as resp:
            if resp.status != 200:
                return []
            data = await resp.json()
            return data.get("sessions", [])

async def fetch_chat_messages(participant_id: str, date: str):
    """Fetch chat messages for a participant on a specific date."""
    headers = {"x-api-key": VAULT_API_KEY}
    url = f"{VAULT_API_URL}/participant/{participant_id}/messages/{date}"
    async with aiohttp.ClientSession() as session:
        async with session.get(url, headers=headers) as resp:
            if resp.status != 200:
                return []
            data = await resp.json()
            return data.get("messages", [])

sessions = await fetch_sessions(yesterday)
print(f"📊 Fetched {len(sessions)} sessions")

In [ ]:
def prepare_training_data(sessions, messages):
    """Prepare training data separated by persona."""
    samara_data = []
    artery_data = []
    
    for session in sessions:
        pid = session["participant_id"]
        study_group = session.get("study_group", "A")
        msgs = messages.get(pid, [])
        
        for i in range(len(msgs) - 1):
            if msgs[i]["role"] == "user" and msgs[i+1]["role"] == "assistant":
                pair = {
                    "input": msgs[i]["content"],
                    "output": msgs[i+1]["content"]
                }
                if study_group == "A":
                    samara_data.append(pair)
                else:
                    artery_data.append(pair)
    
    return samara_data, artery_data

In [ ]:
def fine_tune_lora(training_data, adapter_name, output_dir):
    """Fine-tune LoRA adapter on the provided data."""
    if len(training_data) < 5:
        print(f"⚠️ Insufficient data for {adapter_name} ({len(training_data)} samples)")
        return
    
    # Load base model
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    tokenizer.pad_token = tokenizer.eos_token
    
    # LoRA configuration
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16 if adapter_name == "samara" else 8,
        lora_alpha=32 if adapter_name == "samara" else 16,
        lora_dropout=0.05 if adapter_name == "samara" else 0.0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] if adapter_name == "samara" else ["q_proj", "v_proj"],
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # Training loop (simplified for notebook)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
    
    for epoch in range(2):
        total_loss = 0
        for pair in training_data:
            inputs = tokenizer(pair["input"], return_tensors="pt", truncation=True, max_length=512)
            labels = tokenizer(pair["output"], return_tensors="pt", truncation=True, max_length=512)
            
            outputs = model(**inputs, labels=labels["input_ids"])
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
        
        print(f"  Epoch {epoch+1}: loss = {total_loss/len(training_data):.4f}")
    
    # Save adapter
    model.save_pretrained(output_dir)
    print(f"✅ Saved {adapter_name} adapter to {output_dir}")
    
    return model

In [ ]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================
all_messages = {}
for session in sessions:
    pid = session["participant_id"]
    all_messages[pid] = await fetch_chat_messages(pid, yesterday)

samara_data, artery_data = prepare_training_data(sessions, all_messages)
print(f"Samara samples: {len(samara_data)}, Artery samples: {len(artery_data)}")

if samara_data:
    fine_tune_lora(samara_data, "samara", "./samara_lora")
if artery_data:
    fine_tune_lora(artery_data, "artery", "./artery_lora")

print(f"\n✅ Nightly consolidation complete for {yesterday}")

In [ ]:
# =============================================================================
# UPLOAD TO HUGGING FACE (Optional)
# =============================================================================
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path="./samara_lora",
    repo_id="project-imace/revarie-samara-lora",
    repo_type="model"
)
api.upload_folder(
    folder_path="./artery_lora",
    repo_id="project-imace/revarie-artery-lora",
    repo_type="model"
)
print("📤 Uploaded adapters to Hugging Face")

## Keep-Alive Cell (Prevents Kaggle Timeout)
Run this cell first to keep the session active during long training runs.

In [ ]:
import threading
import time

def keep_alive():
    while True:
        time.sleep(300)
        print(f"⏰ Keep-alive ping at {datetime.utcnow().isoformat()}")

threading.Thread(target=keep_alive, daemon=True).start()